# Exploratory Data Analysis

In [1]:
import pandas as pd

We load the Data:

In [2]:
df_train = pd.read_parquet('../data/train.parquet')
df_test = pd.read_parquet('../data/test.parquet')
df_sub = pd.read_csv('../data/example_submission.csv')

## df_train

In [3]:
print(f"Train shape : {df_train.shape} | Users uniques : {df_train['userId'].nunique()}")
display(df_train.head())

summary_train = pd.DataFrame({
    'dtype': df_train.dtypes,
    'nunique': df_train.nunique(),
    'missing': df_train.isna().sum(),
    'missing_%': (df_train.isna().mean() * 100).round(2)
})
display(summary_train)

Train shape : (17499636, 19) | Users uniques : 19140


,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
0,200,M,Shlok,paid,Johnson,1749042,1538352001000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",278,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,524.32934,Ich mache einen Spiegel - Dream Part 4,Popol Vuh,2018-10-01 00:00:01,2018-08-08 13:22:21
992,200,M,Shlok,paid,Johnson,1749042,1538352525000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",279,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,178.02404,Monster (Album Version),Skillet,2018-10-01 00:08:45,2018-08-08 13:22:21
1360,200,M,Shlok,paid,Johnson,1749042,1538352703000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",280,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,232.61995,Seven Nation Army,The White Stripes,2018-10-01 00:11:43,2018-08-08 13:22:21
1825,200,M,Shlok,paid,Johnson,1749042,1538352935000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",281,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,265.50812,Under The Bridge (Album Version),Red Hot Chili Peppers,2018-10-01 00:15:35,2018-08-08 13:22:21
2366,200,M,Shlok,paid,Johnson,1749042,1538353200000,Logged In,NextSong,22683,"Dallas-Fort Worth-Arlington, TX",282,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,471.69261,Circlesong 6,Bobby McFerrin,2018-10-01 00:20:00,2018-08-08 13:22:21


,dtype,nunique,missing,missing_%
status,int64,3,0,0.00
gender,str,2,0,0.00
firstName,str,4967,0,0.00
level,str,2,0,0.00
lastName,str,1000,0,0.00
userId,str,19140,0,0.00
ts,int64,4189091,0,0.00
auth,str,2,0,0.00
page,str,19,0,0.00
sessionId,int64,161194,0,0.00


### First observations (`df_train`):
- 17.5M event logs for 19,140 unique users (~914 events/user on average) -> we will need to aggregate at the `userId` level.
- No missing values on 16/19 columns. The exact 18.33% missing values on `song`, `artist`, and `length` correspond to non-song events (clicks, ads, thumbs, etc.) -> we must NOT drop NaNs.
- Low-cardinality columns (`page`: 19, `auth`: 2, `level`: 2, `gender`: 2, `status`: 3) are easy to exploit; `location` (875) and `userAgent` (85) will need parsing; `firstName`/`lastName` can be ignored.
- `time` and `registration` are already parsed as `datetime64[us]`.

## df_test

In [4]:
print(f"Test shape : {df_test.shape} | Users uniques : {df_test['userId'].nunique()}")
display(df_test.head())

summary_test = pd.DataFrame({
    'dtype': df_test.dtypes,
    'nunique': df_test.nunique(),
    'missing': df_test.isna().sum(),
    'missing_%': (df_test.isna().mean() * 100).round(2)
})
display(summary_test)

Test shape : (4393179, 19) | Users uniques : 2904


,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
7,200,M,Jonathan,free,Martin,1465194,1538352006000,Logged In,NextSong,22483,"New York-Newark-Jersey City, NY-NJ-PA",29,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,250.82730,Mockingbird,Eminem,2018-10-01 00:00:06,2018-09-27 17:29:36
54,200,M,Jonathan,free,Martin,1465194,1538352028000,Logged In,Roll Advert,22483,"New York-Newark-Jersey City, NY-NJ-PA",30,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",GET,NaN,NaN,NaN,2018-10-01 00:00:28,2018-09-27 17:29:36
477,200,M,Jonathan,free,Martin,1465194,1538352256000,Logged In,NextSong,22483,"New York-Newark-Jersey City, NY-NJ-PA",31,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,355.78730,Thank You (Precious Memories Album Version),Ray Boltz,2018-10-01 00:04:16,2018-09-27 17:29:36
1170,200,M,Jonathan,free,Martin,1465194,1538352611000,Logged In,NextSong,22483,"New York-Newark-Jersey City, NY-NJ-PA",32,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,191.68608,Mathletics,Foals,2018-10-01 00:10:11,2018-09-27 17:29:36
1552,200,M,Jonathan,free,Martin,1465194,1538352802000,Logged In,NextSong,22483,"New York-Newark-Jersey City, NY-NJ-PA",33,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",PUT,275.25179,Proceed,The Roots,2018-10-01 00:13:22,2018-09-27 17:29:36


,dtype,nunique,missing,missing_%
status,int64,3,0,0.00
gender,str,2,653681,14.88
firstName,str,1479,653681,14.88
level,str,2,0,0.00
lastName,str,797,653681,14.88
userId,str,2904,0,0.00
ts,int64,2698359,0,0.00
auth,str,3,0,0.00
page,str,20,0,0.00
sessionId,int64,145863,0,0.00


### Comparison with `df_test`:
- 4.39M event logs for 2,904 unique users (~1,513 events/user on average, much higher than the ~914 in train).
- Major difference: `gender`, `firstName`, `lastName`, `location`, and `userAgent` have exactly 653,681 missing values (14.88%) in `df_test` (vs 0% in `df_train`).
- Missing values on `song`, `artist`, and `length` also jump from 18.33% (train) to 30.31% (test).
- `auth` (3 vs 2) and `page` (20 vs 19) do not have the same number of categories between train and test -> we need to inspect their exact values to understand this shift.

## Comparing auth, page, and IDs (train vs test)

In [5]:
display(pd.DataFrame({
    'train': df_train['auth'].value_counts(),
    'test': df_test['auth'].value_counts()
}).fillna(0).astype(int))

display(pd.DataFrame({
    'train': df_train['page'].value_counts(),
    'test': df_test['page'].value_counts()
}).fillna(0).astype(int))

,train,test
auth,,
Cancelled,4271,0
Guest,0,3363
Logged In,17495365,3739498
Logged Out,0,650318


,train,test
page,,
About,33117,44486
Add Friend,262147,56431
Add to Playlist,409606,87448
Cancel,4271,0
Cancellation Confirmation,4271,0
Downgrade,124248,26989
Error,17294,4445
Help,89035,40651
Home,645259,481816


In [6]:
users_communs = set(df_train['userId'].unique()) & set(df_test['userId'].unique())
print("Users en commun train/test :", len(users_communs))
print("Shape df_sub :", df_sub.shape)
display(df_sub.head())

Users en commun train/test : 0
Shape df_sub : (2904, 2)


,id,target
0,1128274,0
1,1782451,1
2,1611542,1
3,1241663,0
4,1653104,1


### Key takeaways from `auth`, `page`, and IDs:
- `Logged Out` (650,318) + `Guest` (3,363) = 653,681 rows in `df_test`, which matches the exact number of missing profile values (`gender`, `location`, `userAgent`, etc.).
- Train-only events: `auth == 'Cancelled'`, `page == 'Cancel'`, and `page == 'Cancellation Confirmation'` (all 4,271 rows = our churn target, must be removed from features to avoid data leakage).
- Test-only events: `Guest`, `Logged Out`, and pages `Login`, `Register`, `Submit Registration` (which also inflate `Home`/`About`/`Help` counts in `df_test`).
- Decision for preprocessing: extract the target `y` from `df_train` first, then filter both `df_train` and `df_test` on `auth == 'Logged In'` (and drop cancellation pages).
- Zero user overlap between train and test, and `df_sub` matches the 2,904 test users (`id, target`).

## Target extraction and cleaning

In [7]:
churn_users = df_train.loc[df_train['page'] == 'Cancellation Confirmation', 'userId'].unique()
all_users = pd.Index(sorted(df_train['userId'].unique()), name='userId')
y_train = pd.Series(all_users.isin(churn_users).astype(int), index= all_users, name='target')
print("Shape y_train :", y_train.shape)
print("Nb churners   :", y_train.sum(), f"({y_train.mean()*100:.2f}%)")
display(y_train.head())

Shape y_train : (19140,)
Nb churners   : 4271 (22.31%)


userId
1000025    1
1000035    0
1000083    1
1000103    0
1000164    0
Name: target, dtype: int64

In [8]:
df_train_clean = df_train[
    (df_train['auth']=='Logged In') & (~df_train['page'].isin(['Cancellation Confirmation', 'Cancel']))
    ]

In [9]:
df_test_clean = df_test[df_test['auth']=='Logged In']

In [10]:
display(pd.DataFrame({
    'train': df_train_clean['auth'].value_counts(),
    'test': df_test_clean['auth'].value_counts()
}).fillna(0).astype(int))

display(pd.DataFrame({
    'train': df_train_clean['page'].value_counts(),
    'test': df_test_clean['page'].value_counts()
}).fillna(0).astype(int))

,train,test
auth,,
Logged In,17491094,3739498


,train,test
page,,
NextSong,14291433,3061811
Thumbs Up,789391,167351
Home,645259,137301
Add to Playlist,409606,87448
Roll Advert,284837,56741
Add Friend,262147,56431
Logout,204700,43696
Thumbs Down,164964,35367
Downgrade,124248,26989


In [11]:
users_communs = set(df_train_clean['userId'].unique()) & set(df_test_clean['userId'].unique())
print("Users en commun train/test :", len(users_communs))
print("Train clean users :", df_train_clean['userId'].nunique())
print("Test clean users  :", df_test_clean['userId'].nunique())
print("Missing gender in test_clean :", df_test_clean['gender'].isna().sum())

Users en commun train/test : 0
Train clean users : 19140
Test clean users  : 2903
Missing gender in test_clean : 0


In [12]:
missing_test_user = set(df_test['userId'].unique()) - set(df_test_clean['userId'].unique())
print("User sans aucun événement Logged In :", missing_test_user)
display(df_test[df_test['userId'].isin(missing_test_user)])

User sans aucun événement Logged In : {'1261737'}


,status,gender,firstName,level,lastName,userId,ts,auth,page,sessionId,location,itemInSession,userAgent,method,length,song,artist,time,registration
17,200,NaN,NaN,paid,NaN,1261737,1538352008000,Logged Out,Home,8615,NaN,87,NaN,GET,NaN,NaN,NaN,2018-10-01 00:00:08,2018-10-01 00:00:08
71,307,NaN,NaN,free,NaN,1261737,1538352041000,Logged Out,Login,7433,NaN,0,NaN,PUT,NaN,NaN,NaN,2018-10-01 00:00:41,2018-10-01 00:00:08
354,200,NaN,NaN,free,NaN,1261737,1538352182000,Logged Out,Home,25003,NaN,4,NaN,GET,NaN,NaN,NaN,2018-10-01 00:03:02,2018-10-01 00:00:08
472,200,NaN,NaN,free,NaN,1261737,1538352254000,Logged Out,Home,9930,NaN,2,NaN,GET,NaN,NaN,NaN,2018-10-01 00:04:14,2018-10-01 00:00:08
476,307,NaN,NaN,free,NaN,1261737,1538352255000,Logged Out,Login,9930,NaN,3,NaN,PUT,NaN,NaN,NaN,2018-10-01 00:04:15,2018-10-01 00:00:08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25661418,200,NaN,NaN,paid,NaN,1261737,1542671712000,Logged Out,Home,37204,NaN,29,NaN,GET,NaN,NaN,NaN,2018-11-19 23:55:12,2018-10-01 00:00:08
25661455,200,NaN,NaN,paid,NaN,1261737,1542671748000,Logged Out,About,37204,NaN,30,NaN,GET,NaN,NaN,NaN,2018-11-19 23:55:48,2018-10-01 00:00:08
25661460,307,NaN,NaN,paid,NaN,1261737,1542671749000,Logged Out,Login,37204,NaN,31,NaN,PUT,NaN,NaN,NaN,2018-11-19 23:55:49,2018-10-01 00:00:08
25661574,200,NaN,NaN,paid,NaN,1261737,1542671906000,Logged Out,Home,37204,NaN,35,NaN,GET,NaN,NaN,NaN,2018-11-19 23:58:26,2018-10-01 00:00:08


### Summary of cleaning:
- We extracted `y_train`: 4,271 churners out of 19,140 users (22.31%).
- After keeping only `auth == 'Logged In'` (and removing `Cancel` and `Cancellation Confirmation`), train and test have the exact same 17 pages with similar proportions (around 4.7x more rows in train across all pages).
- All 653,681 `Logged Out` / `Guest` rows in `df_test` actually belong to one single userId (`1261737`), which explains all the missing values in `gender`, `location`, etc. Once removed, there are 0 missing profile values left.
- Since `1261737` is still in `example_submission.csv`, we will have to keep it in our final submission (and probably predict 0 since it never logged in).

## Temporal analysis (checking time bias)

In [13]:
train_time_stats = df_train_clean.groupby('userId')['time'].agg(['min', 'max', 'count'])
train_time_stats['target'] = y_train

test_time_stats = df_test_clean.groupby('userId')['time'].agg(['min', 'max', 'count'])

display(train_time_stats.groupby('target').median())
display(test_time_stats.median().to_frame('test'))

,min,max,count
target,,,
0,2018-10-04 10:16:24,2018-11-15 19:39:16,531.0
1,2018-10-02 09:04:21,2018-10-23 02:13:19,555.0


,test
min,2018-10-03 08:58:35
max,2018-11-17 00:04:56
count,835.0


### Observations on time and volume:
- In `train`, non-churners stay active until around Nov 15 (median), while churners stop around Oct 23 because their logs end when they cancel.
- Even though churners are observed for half as many days (~21 vs ~42 days), they have a slightly higher median event count (555 vs 531), meaning higher daily intensity.
- In `test`, users are observed until around Nov 17 (median) with a much higher median count (835 events).
- Conclusion: we must not use `max(time)`, observation duration, or raw counts as features (otherwise the model would learn the early cutoff of churners in `train` and fail on `test`). We will use ratios, averages per active day / session, and trends instead.